## Web検索連動チャットボットを作成しよう

In [5]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ===== グラフの構築 =====
def build_graph(model_name):
    # 1) LLM を用意（ツール呼び出しを有効化）
    llm = ChatOpenAI(model=model_name, temperature=0)

    # 2) Web検索ツール（Tavily）
    search_tool = TavilySearchResults(max_results=5)
    tools = [search_tool]

    # 3) ツールノード（LLMがtool_callsしたらここで実行される）
    tool_node = ToolNode(tools)

    # 4) アシスタントノード（LLMにメッセージを渡して応答を作る）
    #    bind_tools しておくと、必要に応じて tool_calls を返すようになります。
    llm_with_tools = llm.bind_tools(tools)

    def assistant(state: State):
        # state["messages"] は add_messages により蓄積される
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    # 5) StateGraph を組み立て
    graph_builder = StateGraph(State)
    graph_builder.add_node("assistant", assistant)
    graph_builder.add_node("tools", tool_node)

    # 開始点 → assistant
    graph_builder.set_entry_point("assistant")

    # assistant の出力に tool_calls があれば tools へ、なければ終了へ
    graph_builder.add_conditional_edges(
        "assistant",
        tools_condition,
        # tools_condition が返すキーに対応する遷移先
        {
            "tools": "tools",
            "__end__": "__end__",
        },
    )

    # tools 実行後は assistant に戻して、検索結果を踏まえて最終回答させる
    graph_builder.add_edge("tools", "assistant")

    # 6) 会話スレッド（thread_id）ごとに状態を保持するためのメモリ
    checkpointer = MemorySaver()

    # 7) コンパイル
    graph = graph_builder.compile(checkpointer=checkpointer)
    return graph

# ===== グラフ実行関数 =====
def stream_graph_updates(graph: StateGraph, user_input: str):
    print(f"質問: {user_input}", flush=True)

    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values")
    # 結果をストリーミングで得る
    for event in events:
        last = event["messages"][-1]
        # tool 実行結果(JSON)を表示しない
        if getattr(last, "type", None) == "ai":
            print(last.content, flush=True)

# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini" 

# グラフの作成
graph = build_graph(MODEL_NAME)

# チャットボットのループ
while True:
    user_input = input("質問:")
    if user_input.strip()=="":
        print("ありがとうございました!")
        break
    stream_graph_updates(graph, user_input)

質問: こんばんわ！
こんばんは！今日はどんなことをお手伝いできますか？
質問: 9ひく5は？
9ひく5は4です。
質問: 台湾観光について検索結果を教えて

台湾観光に関する情報をいくつかご紹介します。

1. **[台湾の基本情報と観光ガイド・有名観光スポット](https://www.arukikata.co.jp/area/tw/)**
   - 台湾の観光スポットや基本情報が詳しく紹介されています。台北の国立故宮博物院や龍山寺、台中の寶覺寺、台南の赤崁楼など、各地の名所が掲載されています。また、台湾グルメや温泉情報も充実しています。

2. **[台湾観光におすすめの名所＆人気のスポットランキング](https://www.hankyu-travel.com/guide/taiwan/)**
   - 台北101や九份、士林夜市など、台湾の人気観光スポットがランキング形式で紹介されています。絶景やグルメ、ショッピング、温泉など、様々な楽しみ方が提案されています。

3. **[台湾・台北観光のおすすめスポット 19選](https://www.knt.co.jp/travelguide/kaigai/027/)**
   - 台北の観光スポットを19選にまとめたガイドです。国立故宮博物院や士林観光夜市、九份など、観光名所が詳しく説明されています。

4. **[台湾観光おすすめ42選！モデルコースや現地スタッフの情報](https://www.kkday.com/ja/blog/48762/taiwan-sightseeing?srsltid=AfmBOopu7ST_fpZJOwvvHfNOdDfh-QP9gHSdd_ib-oYuY3143R13Uox1)**
   - 台湾の観光スポットを42選にまとめた情報です。台北101や龍山寺、士林夜市などの定番スポットから、九份や淡水などの近郊スポットまで幅広く紹介されています。

5. **[台北観光サイト](https://www.travel.taipei/ja)**
   - 台北の公式観光サイトで、観光名所やイベント情報、交通情報などが掲載されています。観光マップやアプリのダウンロードも可能です。

これらの情報を参考に、台湾観光を計画してみてください！
質問: 4番目のおすすめTOP3は？
4番目の「